# Tarea 1 — Clustering de transacciones con productos anulados
**Métodos de aprendizaje de máquinas en Data Science** — MSc Data Science, UAI
Profesor: Sebastián Moreno

---

### Modelos elegidos

**K-means** (Problema 2, clase 03) y **DBSCAN** (Problema 3, clase 07).

| Modelo | Por qué |
|---|---|
| **K-means** | Método de partición. Centroides interpretables en unidades de negocio. |
| **DBSCAN** | Método de densidad. Etiqueta ruido con `-1`, que en un caso de fraude es el grupo de interés. No impone forma esférica ni número de clusters. |
| Jerárquico (clase 04) | Descartado: O(n²) en memoria. Con 2,4 M de filas la matriz de distancias exigiría del orden de 23 TB. |
| GMM (clase 05) | Descartado: es K-means con asignación blanda. Compararlo contra K-means en el Problema 4 sería contrastar dos variantes de la misma lógica de partición. |

### Alcance metodológico

Este notebook usa **únicamente** técnicas del material del curso:

- Limpieza, imputación con mediana, construcción de variables, transformaciones Box-Cox y
  arcsine, y estandarización (clase 02)
- K-means, inercia, eliminación de variables correlacionadas (clase 03)
- Estadístico de Hopkins, coeficiente de silhouette, selección de K por codo o peak sobre
  SSE y Silhouette, evaluación supervisada con Rand index, NMI y
  homogeneidad/completitud/V-measure (clase 06)
- DBSCAN, `eps`, `MinPts`, puntos core, curva de distancia al k-ésimo vecino (clase 07)

No se usan librerías fuera de las que Colab ya trae, ni métricas o transformaciones ajenas al curso.

### Nota sobre la columna `Fraude`

La base trae una columna `Fraude` que no figura en el anexo del enunciado. **No se usa para construir
los clusters**, porque el problema es de aprendizaje no supervisado. Se reserva para la evaluación
supervisada de la clase 06, en las secciones 2.7, 3.5 y 4.2.

## 0. Preparación del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import csv, gc, os, time, warnings

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             rand_score, normalized_mutual_info_score,
                             homogeneity_completeness_v_measure)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
print('Entorno listo. Sin instalaciones externas.')

### 0.1 Carga del dataset desde Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

NOMBRE = 'data-tarea1.csv'
FILE_ID = '1_UCz0bUBc2YoDYKi6oSYAwommyv_EDA5'

RUTA = None
for base, _, files in os.walk('/content/drive/MyDrive'):
    if NOMBRE in files:
        RUTA = os.path.join(base, NOMBRE)
        break

if RUTA is None:
    print('No se encontró en Drive. Descargando...')
    os.system('pip -q install gdown')
    os.system(f'gdown {FILE_ID} -O /content/data-tarea1.csv')
    RUTA = '/content/data-tarea1.csv'

print('Archivo:', RUTA)
print('Tamaño: %.1f MB' % (os.path.getsize(RUTA)/1024**2))

### 0.2 Inspección previa

In [ ]:
with open(RUTA, 'r', encoding='utf-8', errors='ignore') as f:
    cabecera = f.readline()
    muestra_txt = cabecera + ''.join([f.readline() for _ in range(50)])

try:
    SEP = csv.Sniffer().sniff(muestra_txt, delimiters=',;|\t').delimiter
except Exception:
    SEP = max([',', ';', '|', '\t'], key=cabecera.count)

print('Separador:', repr(SEP), '| Columnas:', len(cabecera.split(SEP)))
muestra = pd.read_csv(RUTA, sep=SEP, nrows=100_000, low_memory=False)
print('Muestra:', muestra.shape)

diag = pd.DataFrame({'dtype': muestra.dtypes.astype(str),
                     'nulos_%': (muestra.isna().mean()*100).round(2),
                     'unicos': muestra.nunique(),
                     'min': muestra.min(numeric_only=True),
                     'max': muestra.max(numeric_only=True)})
print(diag.to_string())

### 0.3 Carga completa

In [ ]:
dtypes = {c: ('int32' if t == 'int64' else 'float32')
          for c, t in muestra.dtypes.items() if t in ('int64', 'float64')}

df = pd.read_csv(RUTA, sep=SEP, dtype=dtypes, low_memory=False)
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed:')], errors='ignore')
print('Dimensiones:', df.shape)
print('Memoria: %.1f MB' % (df.memory_usage(deep=True).sum()/1024**2))
del muestra; _ = gc.collect()
df.head()

---
# Problema 1 — Limpieza y selección de variables (1 pto)

Cada decisión se justifica. Lo descartado aquí no entra a los modelos, pero se conserva en `df`
para **describir** los clusters después, tal como permite el enunciado.

### 1.1 Clasificación de las variables

Los identificadores no entran a un modelo de distancia: `TerminalId = 5` no está "más cerca" de
`TerminalId = 6` que de `TerminalId = 200`, el número es una etiqueta.

La columna `Fraude` se aparta de inmediato: es una etiqueta y usarla para construir clusters sería
resolver el problema mirando la respuesta.

In [ ]:
GRUPOS_DEF = {
 'IDENTIFICADORES': ['ID_Supervisor','SupervisorCod','LocalFisicoCod','TerminalId','Trx',
                     'EstadoTrxCod','Llave'],
 'TIEMPO'         : ['FechaIniTrx','HoraIniTrx','MinutoIniTrx','SegundoIniTrx',
                     'HoraFinTrx','MinutoFinTrx','SegundoFinTrx'],
 'CANTIDADES'     : ['CantDiferenteSku','CantItem','CantItemAnulado','CantItemOriginal',
                     'CantItemTrx','CantLinea','CantLineaAnulada','CantTotalItem','Q_anulado_aggr'],
 'MONTOS'         : ['MontoTicket','MontoCmrDebito','MontoCmrCredito','MontoOtroCredito',
                     'MontoOtroDebito','MontoEfectivo','MontoOtroMedioPago',
                     'MontoDescuentoPMM','Monto_Anul'],
 'CATEGORIAS_PROD': ['q_jugos','q_Envases','q_Energeticas','q_Vacuno','q_Formulas','q_Mascota'],
 'PORCENTAJES'    : ['porc_anulado_envase','porc_anulado_jugo'],
 'DUMMIES'        : ['q_Override','MontoCmrDebito_Dummy','MontoCmrCredito_Dummy',
                     'MontoOtroCredito_Dummy','MontoOtroDebito_Dummy','MontoEfectivo_Dummy',
                     'MontoOtroMedioPago_Dummy','MontoDescuentoPMM_Dummy'],
}

ETIQUETA = 'Fraude' if 'Fraude' in df.columns else None
if ETIQUETA:
    print(f'Etiqueta externa: {ETIQUETA}')
    print(df[ETIQUETA].value_counts().sort_index().to_string())
    print('NO entra al modelo. Se usa para validar (secciones 2.7, 3.5 y 4.2).\n')

grupos = {k: [c for c in v if c in df.columns] for k, v in GRUPOS_DEF.items()}
for k, v in grupos.items():
    faltan = set(GRUPOS_DEF[k]) - set(v)
    print(f'{k:17s} {len(v):2d}/{len(GRUPOS_DEF[k]):2d}' +
          (f'   faltan: {sorted(faltan)}' if faltan else ''))

declaradas = {c for v in GRUPOS_DEF.values() for c in v} | ({ETIQUETA} if ETIQUETA else set())
extra = [c for c in df.columns if c not in declaradas]
if extra: print('\nColumnas no previstas:', extra)

const = [c for c in df.columns if c != ETIQUETA and df[c].nunique(dropna=False) <= 1]
if const:
    print('\nConstantes (sin información), se descartan:', const)
    df = df.drop(columns=const)
    grupos = {k: [c for c in v if c in df.columns] for k, v in grupos.items()}
print('\nDimensiones:', df.shape)

### 1.2 Duplicados

In [ ]:
n0 = len(df)
df = df.drop_duplicates()
print('Duplicados exactos eliminados:', n0 - len(df))

if 'Llave' in df.columns:
    dup = df['Llave'].duplicated().sum()
    print('Llaves repetidas:', dup)
    if dup > 0:
        df = df.drop_duplicates(subset='Llave', keep='first')
print('Dimensiones:', df.shape)

### 1.3 Valores perdidos

La clase 02 plantea tres opciones: eliminar el dato o la variable, estimar el valor, o dejarlo
vacío. Y advierte explícitamente que hay que tener cuidado al reemplazar por media o mediana.

Criterio adoptado:
1. Variable con más de 40% de nulos: **se elimina la variable**. Imputar la mayoría del contenido
   sería inventar señal.
2. Porcentajes de anulación nulos: **se imputan con 0**, porque ahí el nulo significa "no hubo
   anulación de esa categoría", no "dato faltante". Es conocimiento de dominio, no una estimación
   estadística.
3. El resto: **mediana**, no media, porque son montos muy asimétricos y la media está arrastrada
   por los extremos. Se reporta cuántos valores se imputan en cada variable para dimensionar el
   efecto.

In [ ]:
nulos = (df.isna().mean()*100).sort_values(ascending=False)
print(nulos[nulos > 0].round(2).to_string() if (nulos > 0).any() else 'Sin valores nulos.')

descartar = nulos[nulos > 40].index.tolist()
if descartar:
    print('\nEliminadas por >40% nulos:', descartar)
    df = df.drop(columns=descartar)
    grupos = {k: [c for c in v if c in df.columns] for k, v in grupos.items()}

for c in grupos['PORCENTAJES']:
    if c in df.columns and df[c].isna().any():
        print(f'{c}: {int(df[c].isna().sum()):,} nulos -> 0 (ausencia de anulación)')
        df[c] = df[c].fillna(0)

for c in df.select_dtypes(include=[np.number]).columns:
    if df[c].isna().any():
        print(f'{c}: {int(df[c].isna().sum()):,} nulos -> mediana ({df[c].median():.2f})')
        df[c] = df[c].fillna(df[c].median())

print('\nNulos restantes:', int(df.isna().sum().sum()))

### 1.4 Consistencia de los datos

Tres revisiones, en orden:

**1. Signos.** Las columnas de anulación vienen negativas por convención contable ("resta al
ticket"). Como *todas* las transacciones tienen al menos un producto anulado, el chequeo
`valor < 0` sin corregir el signo eliminaría el 100% de las filas.

**2. Rango.** Horas entre 0 y 23, minutos y segundos entre 0 y 59, cantidades y montos no
negativos, porcentajes dentro de su dominio.

**3. Coherencia entre variables.** No basta con que cada variable esté en rango: tienen que ser
consistentes entre sí. No se pueden anular más ítems de los que pasaron por la caja.

Un chequeo que descarte más del 5% de las filas no se aplica y se reporta para revisión: a esa
escala el problema está en el criterio, no en los datos. Se verifica además que el conjunto de
filtros no elimine más del 7%.

In [ ]:
# 1) Convención de signo
NEG_CONV = [c for c in ['Monto_Anul','MontoDescuentoPMM','CantItemAnulado',
                        'CantLineaAnulada','Q_anulado_aggr']
            if c in df.columns and df[c].min() < 0]
for c in NEG_CONV:
    print(f'{c}: [{df[c].min():.2f}, {df[c].max():.2f}] -> valor absoluto')
    df[c] = df[c].abs()

# Negativos pequeños en montos de pago: redondeos o vueltos, no errores de la transacción
TOL_NEG = 100
for c in grupos['MONTOS']:
    if c in df.columns:
        chicos = df[c].between(-TOL_NEG, 0, inclusive='left')
        if chicos.any():
            print(f'{c}: {int(chicos.sum()):,} negativos pequeños (>= -{TOL_NEG}) -> 0')
            df.loc[chicos, c] = 0

# 2) y 3) Rango y coherencia
UMBRAL_IND, UMBRAL_ACUM = 5.0, 7.0
mask_ok = pd.Series(True, index=df.index)
rep = []

def aplicar(nombre, mala):
    global mask_ok
    pct = mala.mean()*100
    if pct > UMBRAL_IND:
        rep.append((nombre, int(mala.sum()), round(pct,2), 'NO aplicado - revisar'))
    else:
        mask_ok &= ~mala
        rep.append((nombre, int(mala.sum()), round(pct,2), 'aplicado'))

for c in ['HoraIniTrx','HoraFinTrx']:
    if c in df.columns: aplicar(c, ~df[c].between(0, 23))
for c in ['MinutoIniTrx','SegundoIniTrx','MinutoFinTrx','SegundoFinTrx']:
    if c in df.columns: aplicar(c, ~df[c].between(0, 59))
for c in grupos['CANTIDADES'] + grupos['MONTOS'] + grupos['CATEGORIAS_PROD']:
    if c in df.columns: aplicar(c, df[c] < 0)
for c in grupos['PORCENTAJES']:
    if c in df.columns:
        tope = 100 if df[c].max() > 1.5 else 1
        aplicar(f'{c} (0-{tope})', ~df[c].between(0, tope))
for anulado, total in [('CantItemAnulado','CantItem'), ('CantLineaAnulada','CantLinea')]:
    if anulado in df.columns and total in df.columns:
        aplicar(f'{anulado} > {total}', df[anulado] > df[total])

t = pd.DataFrame(rep, columns=['variable','filas_malas','%','estado'])
print('\n' + (t[t.filas_malas > 0].to_string(index=False) if (t.filas_malas > 0).any()
               else 'Ningún chequeo encontró filas inválidas.'))

pct_total = (~mask_ok).mean()*100
print('\nA eliminar: %d (%.4f%%)' % ((~mask_ok).sum(), pct_total))
assert mask_ok.sum() > 0, 'El filtro dejaría 0 filas.'
assert pct_total <= UMBRAL_ACUM, f'Los filtros descartan {pct_total:.2f}%, sobre el {UMBRAL_ACUM}%.'

df = df[mask_ok].reset_index(drop=True)
print('Dimensiones:', df.shape)

### 1.5 Construcción de variables

La clase 02 define feature engineering como transformar datos crudos en variables que representen
mejor el problema. Se construyen **cinco**, cada una porque las columnas originales no expresan eso.

| Variable | Por qué |
|---|---|
| `ratio_item_anulado` | Sin ella el modelo agrupa por **tamaño de compra**. Un ticket de $200.000 con 2 anulaciones queda lejísimos de uno de $5.000 con 2 anulaciones, cuando la conducta es la misma. |
| `ratio_monto_anulado` | Lo mismo en pesos: distingue anular un chicle de anular un vacuno. |
| `duracion_seg` | La hora de inicio y la de fin por separado no dicen cuánto demoró. |
| `hora_decimal` | Une hora y minuto en un solo eje continuo en vez de dos columnas. |
| `ratio_efectivo` | El efectivo es el vector clásico de este fraude. El monto absoluto se confunde con el tamaño del ticket. |

No se construye nada más. Cada dimensión adicional deteriora la separación en el espacio euclidiano,
así que agregar variables "por si acaso" tiene costo.

In [ ]:
eps_div = 1e-9

if all(c in df.columns for c in ['HoraIniTrx','MinutoIniTrx','SegundoIniTrx',
                                 'HoraFinTrx','MinutoFinTrx','SegundoFinTrx']):
    ini = df['HoraIniTrx']*3600 + df['MinutoIniTrx']*60 + df['SegundoIniTrx']
    fin = df['HoraFinTrx']*3600 + df['MinutoFinTrx']*60 + df['SegundoFinTrx']
    dur = (fin - ini).astype('float32')
    cruce = dur < 0
    dur[cruce] += 86400                      # la transacción cruzó medianoche
    print(f'Cruces de medianoche corregidos: {int(cruce.sum()):,}')
    df['duracion_seg'] = dur

if all(c in df.columns for c in ['HoraIniTrx','MinutoIniTrx']):
    df['hora_decimal'] = (df['HoraIniTrx'] + df['MinutoIniTrx']/60).astype('float32')

if all(c in df.columns for c in ['CantItemAnulado','CantItem']):
    df['ratio_item_anulado'] = (df['CantItemAnulado'] /
                                (df['CantItem'].abs() + eps_div)).clip(0, 1).astype('float32')

if all(c in df.columns for c in ['Monto_Anul','MontoTicket']):
    base = df['MontoTicket'].abs() + df['Monto_Anul'].abs() + eps_div
    df['ratio_monto_anulado'] = (df['Monto_Anul'].abs() / base).clip(0, 1).astype('float32')

if all(c in df.columns for c in ['MontoEfectivo','MontoTicket']):
    df['ratio_efectivo'] = (df['MontoEfectivo'].abs() /
                            (df['MontoTicket'].abs() + eps_div)).clip(0, 1).astype('float32')

NUEVAS = [c for c in ['duracion_seg','hora_decimal','ratio_item_anulado',
                      'ratio_monto_anulado','ratio_efectivo'] if c in df.columns]
print('Creadas:', NUEVAS)
df[NUEVAS].describe().T.round(3)

### 1.6 Outliers y distribuciones sesgadas

La clase 02 dice que **todo outlier debe ser analizado antes de tomar una decisión**, y propone los
métodos visuales (gráfico de caja) para detectarlos. Se analizan primero y después se decide.

Y da el diagnóstico exacto de lo que pasa aquí:

> *"Mientras algunos modelos no son afectados por datos atípicos o la escala de los valores de las
> variables continuas, otros modelos sufren con ello... En el caso de distribuciones sesgadas, la
> cola de la distribución puede afectar negativamente el modelo."*

Los montos de un supermercado son log-normales: la mayoría de los tickets son chicos y unos pocos
son enormes. Si se estandariza sin corregir el sesgo, esos pocos extremos concentran casi toda la
varianza y K-means gasta un centroide en un puñado de transacciones, dejando al resto aplastado en
una sola masa. **El coeficiente de silhouette sube en ese escenario**, porque un cluster de muy
pocos puntos alejados tiene silhouette casi perfecto, así que el número engaña: hay que mirar
también el tamaño del cluster más chico.

La clase 02 propone dos transformaciones que resuelven esto, y se aplican las dos:

**1. Box-Cox** para las variables sesgadas. La lámina la define como la transformación que
*"permite eliminar el sesgo de la distribución"*, con λ como parámetro: λ=1 es no transformar,
λ=0,5 es raíz cuadrada, λ=-1 es la función inversa y **λ=0 es el logaritmo**. El λ se estima de
los datos. Se aplica sobre `x + 1` para garantizar valores estrictamente positivos, que es lo que
requiere Box-Cox.

**2. Arcsine** para las variables que ya están entre 0 y 1. La clase la propone justamente para
ese caso, y explica cuándo sirve: *"cuando se requiere que los valores estén en la misma escala.
Por ejemplo, el cálculo de una distancia"*. Es exactamente nuestra situación, porque el clustering
se basa en distancias.

Con esto **no se elimina ninguna transacción**: los extremos se conservan, que es lo correcto en un
caso de fraude donde son la señal buscada, pero dejan de dominar la geometría del espacio.

In [ ]:
CONTINUAS = [c for c in (grupos['CANTIDADES'] + grupos['MONTOS'] +
                         grupos['CATEGORIAS_PROD'] + NUEVAS) if c in df.columns]

# --- Análisis visual previo (clase 02) ---
sub = df[CONTINUAS].sample(min(50_000, len(df)), random_state=RANDOM_STATE)
n_c = len(CONTINUAS); filas = (n_c + 3)//4
fig, ax = plt.subplots(filas, 4, figsize=(16, 2.6*filas))
for a, c in zip(ax.ravel(), CONTINUAS):
    a.boxplot(sub[c].dropna(), widths=.5)
    a.set_title(c, fontsize=8); a.tick_params(labelsize=7)
for a in ax.ravel()[n_c:]: a.axis('off')
plt.suptitle('Gráficos de caja antes de transformar (clase 02)', y=1.0)
plt.tight_layout(); plt.show()
del sub; _ = gc.collect()

# --- Magnitud del sesgo ---
sesgo_antes = df[CONTINUAS].skew()
print('--- Asimetría antes de transformar ---')
print(sesgo_antes.round(2).sort_values(ascending=False).to_string())

# Se guardan los valores originales para poder describir los clusters en pesos y unidades reales
NUCLEO_PREV = [c for c in ['ratio_item_anulado','ratio_monto_anulado','CantItemAnulado',
                           'Monto_Anul','MontoTicket','CantItem','duracion_seg',
                           'hora_decimal','ratio_efectivo'] if c in df.columns]
df_real = df[NUCLEO_PREV].copy()

# --- 1) Arcsine en las variables acotadas entre 0 y 1 ---
ACOTADAS = [c for c in CONTINUAS
            if (c.startswith('ratio_') or c.startswith('porc_'))
            and df[c].min() >= 0 and df[c].max() <= 1.0001]
for c in ACOTADAS:
    df[c] = np.arcsin(np.sqrt(df[c].clip(0, 1))).astype('float32')
print('\nArcsine aplicada a:', ACOTADAS)

# --- 2) Box-Cox en las sesgadas ---
SESGADAS = [c for c in CONTINUAS
            if c not in ACOTADAS and c != 'hora_decimal'
            and abs(sesgo_antes[c]) > 1 and df[c].min() >= 0]
if SESGADAS:
    pt = PowerTransformer(method='box-cox', standardize=False)
    df[SESGADAS] = pt.fit_transform(df[SESGADAS] + 1).astype('float32')
    print('\nBox-Cox aplicada a', len(SESGADAS), 'variables.')
    print('Lambda estimado por variable (0 = logaritmo, 1 = sin transformación):')
    for c, lam in zip(SESGADAS, pt.lambdas_):
        print(f'   {c:26s} lambda = {lam:+.3f}')
else:
    pt = None
    print('\nNinguna variable requiere Box-Cox.')

print('\n--- Asimetría después ---')
comp = pd.DataFrame({'antes': sesgo_antes, 'despues': df[CONTINUAS].skew()}).round(2)
print(comp.to_string())
print('\nUna asimetría cercana a 0 indica distribución simétrica. Si alguna sigue muy alta,')
print('probablemente sea una variable con muchos ceros, y eso se discute en los resultados.')

### 1.7 Selección de variables

Tres filtros:

**1. Fuera los identificadores.** Son etiquetas, no magnitudes.

**2. Fuera las dummies binarias.** Una variable 0/1 estandarizada aporta contraste máximo sin
gradiente y domina la distancia euclidiana. Se conservan para describir los clusters.

**3. Fuera una de cada par con correlación alta.** La clase 03 advierte que las variables
redundantes pesan doble en la distancia y terminan "acercando puntos que no son cercanos".

Se define un núcleo explícito de 9 variables que responden a la pregunta del negocio: *¿cómo se
comporta esta transacción respecto de la anulación de productos?* Todo lo demás se conserva en `df`
para describir los clusters en 2.4, que es donde aporta.

In [ ]:
NUCLEO = ['ratio_item_anulado','ratio_monto_anulado','CantItemAnulado','Monto_Anul',
          'MontoTicket','CantItem','duracion_seg','hora_decimal','ratio_efectivo']

candidatas = [c for c in NUCLEO if c in df.columns]
faltan = [c for c in NUCLEO if c not in df.columns]
if faltan: print('No disponibles:', faltan)

PARA_PERFILAR = [c for c in df.columns
                 if c not in candidatas and c not in grupos['IDENTIFICADORES']
                 and c not in ('FechaIniTrx', ETIQUETA)
                 and pd.api.types.is_numeric_dtype(df[c])]

var = df[candidatas].var()
sin_var = var[var < 1e-6].index.tolist()
if sin_var:
    print('Sin varianza:', sin_var)
    candidatas = [c for c in candidatas if c not in sin_var]

m_corr = df[candidatas].sample(min(200_000, len(df)), random_state=RANDOM_STATE).corr().abs()
alto = np.triu(np.ones(m_corr.shape), k=1).astype(bool) & (m_corr.values > 0.85)
fuera = set()
for i, j in zip(*np.where(alto)):
    a, b = m_corr.columns[i], m_corr.columns[j]
    if a in fuera or b in fuera: continue
    if a.startswith('ratio_') and not b.startswith('ratio_'):   elim = b
    elif b.startswith('ratio_') and not a.startswith('ratio_'): elim = a
    else: elim = a if df[a].var() < df[b].var() else b
    fuera.add(elim)
    print(f'corr({a}, {b}) = {m_corr.iloc[i, j]:.3f}  ->  fuera {elim}')

FEATURES = [c for c in candidatas if c not in fuera]
assert ETIQUETA not in FEATURES, 'La etiqueta no puede entrar al modelo.'
print(f'\n=== {len(FEATURES)} VARIABLES PARA MODELAR ===')
for c in FEATURES: print('  -', c)
print(f'\n{len(PARA_PERFILAR)} variables reservadas para describir los clusters.')

plt.figure(figsize=(8, 6.5))
sns.heatmap(df[FEATURES].sample(min(200_000, len(df)), random_state=RANDOM_STATE).corr(),
            cmap='RdBu_r', center=0, square=True, linewidths=.4, annot=True, fmt='.2f',
            annot_kws={'size': 7}, cbar_kws={'shrink': .7})
plt.title('Correlación entre las variables finales')
plt.tight_layout(); plt.show()

### 1.8 Estandarización

La clase 02 lo plantea con un ejemplo directo: si una variable está en millones y otra en decenas,
la de mayor escala domina la distancia. Aquí `MontoTicket` está en decenas de miles y
`ratio_item_anulado` entre 0 y 1. Sin estandarizar, el ratio sería irrelevante para el modelo.

No se aplica PCA. Los modelos trabajan sobre las variables originales, que es lo que permite
explicar los clusters en términos de negocio. PCA se usa una sola vez, en 2.5, para dibujar en dos
dimensiones.

In [ ]:
X = df[FEATURES].astype('float32').values
assert X.shape[0] > 0, 'df quedó vacío: revisar la sección 1.4.'

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')
print('Matriz de modelado:', X_scaled.shape)
print('Media tras estandarizar :', np.abs(X_scaled.mean(axis=0)).max().round(6))
print('Desviación tras estand. :', X_scaled.std(axis=0).mean().round(4))

# Muestra de trabajo para lo que no escala a millones de filas
N_MUESTRA = 50_000
idx_m = np.random.choice(len(X_scaled), min(N_MUESTRA, len(X_scaled)), replace=False)
X_m = X_scaled[idx_m]
print('Muestra de trabajo:', X_m.shape)

### 1.9 Estadístico de Hopkins

Antes de elegir K conviene preguntar si los datos tienen tendencia al agrupamiento. Si `H` ronda
0,5 los datos son esencialmente uniformes y cualquier clustering dará silhouette bajo. Sobre 0,75
indica tendencia clara.

Una advertencia para el informe: Hopkins mide desviación respecto de una distribución uniforme.
Variables muy concentradas en un valor, como `ratio_efectivo` que es cero en la mayoría de las
transacciones, inflan el estadístico sin que existan grupos separables. Conviene citarlo como
evidencia de que los datos no son uniformes, que es lo que realmente mide.

In [ ]:
def hopkins(X, m=None, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    m = m or int(0.05 * n)
    nbrs = NearestNeighbors(n_neighbors=2).fit(X)
    idx = rng.choice(n, m, replace=False)
    w = nbrs.kneighbors(X[idx], n_neighbors=2)[0][:, 1]      # real -> vecino más cercano
    U = rng.uniform(X.min(axis=0), X.max(axis=0), (m, d))
    u = nbrs.kneighbors(U, n_neighbors=1)[0][:, 0]           # uniforme -> real más cercano
    return u.sum() / (u.sum() + w.sum())

sub_h = X_m[np.random.choice(len(X_m), min(10_000, len(X_m)), replace=False)]
H = hopkins(sub_h)
print(f'Estadístico de Hopkins: H = {H:.3f}')
print('  -> tendencia clara al agrupamiento' if H > 0.75
      else '  -> estructura débil: esperar silhouette bajo y reportarlo')

---
# Problema 2 — K-means (2 ptos)

### 2.1 Selección del número de clusters

La clase 06 lo plantea así: evaluar una medida específica (Silhouette, SSE o BIC) sobre un rango de
K, y **mirar por un peak, una bajada, un mínimo o un codo**. BIC corresponde a modelos de mezcla,
así que aquí se usan las dos que aplican a K-means: **SSE (inercia)** y **Silhouette**.

La búsqueda se hace sobre la muestra de trabajo. El modelo final sí se ajusta sobre todas las
transacciones.

La decisión es visual, como indica el curso. Si las dos medidas apuntan a valores distintos, se
analiza el rango que sugieren y se elige justificando el criterio, no el número.

In [ ]:
Ks = range(2, 11)
res = []
for k in Ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, max_iter=300).fit(X_m)
    res.append({'K': k, 'SSE': km.inertia_,
                'silhouette': silhouette_score(X_m, km.labels_, sample_size=10_000,
                                               random_state=RANDOM_STATE)})
    print(f"K={k}  SSE={res[-1]['SSE']:12.1f}  silhouette={res[-1]['silhouette']:.4f}")

met = pd.DataFrame(res)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].plot(met.K, met.SSE, 'o-')
ax[0].set_title('SSE total — buscar el codo'); ax[0].set_xlabel('K'); ax[0].grid(alpha=.3)
ax[0].set_xticks(list(Ks))

ax[1].plot(met.K, met.silhouette, 'o-', color='darkorange')
k_peak = int(met.loc[met.silhouette.idxmax(), 'K'])
ax[1].axvline(k_peak, ls='--', c='crimson', alpha=.7)
ax[1].text(k_peak, met.silhouette.max(), f' peak K={k_peak}', color='crimson', va='top')
ax[1].set_title('Coeficiente de Silhouette — buscar el peak')
ax[1].set_xlabel('K'); ax[1].grid(alpha=.3); ax[1].set_xticks(list(Ks))

plt.tight_layout(); plt.show()
print(met.round(4).to_string(index=False))
print('\nLeer los dos gráficos y decidir. El SSE siempre baja: lo que se busca es el punto')
print('donde deja de bajar rápido. El Silhouette se lee por su valor más alto.')

> **Decisión sobre K.** Fijar el valor abajo después de mirar los dos gráficos y escribir la
> justificación. Se evalúa el criterio, no el número.
>
> Si el codo del SSE y el peak del Silhouette no coinciden, el curso sugiere analizar los valores
> intermedios: en el ejemplo de la clase 06, WCD sugería entre 3 y 5 y Silhouette entre 2 y 4, y la
> conclusión fue analizar 3 y 4. Aquí conviene el mismo razonamiento, sumando el criterio de
> negocio: cuántos grupos distintos se pueden explicar de verdad a un usuario no técnico.

In [ ]:
K_KMEANS = 4          # <-- AJUSTAR tras mirar los gráficos de 2.1

t0 = time.time()
kmeans = KMeans(n_clusters=K_KMEANS, random_state=RANDOM_STATE,
                n_init=10, max_iter=300).fit(X_scaled)
df['cluster_km'] = kmeans.labels_
print(f'KMeans ajustado en {time.time()-t0:.1f} s sobre {len(X_scaled):,} filas.')

tam = df['cluster_km'].value_counts().sort_index()
print('\nDistribución de clusters:')
print(pd.DataFrame({'n': tam, '%': (tam/len(df)*100).round(2)}).to_string())
sil = silhouette_score(X_m, kmeans.predict(X_m), sample_size=20_000, random_state=RANDOM_STATE)
print('\nSilhouette (muestra): %.4f' % sil)

# El silhouette por sí solo engaña: un cluster de muy pocos puntos alejados lo infla.
menor = tam.min()/len(df)*100
if menor < 1:
    print(f'\nATENCIÓN: el cluster más chico tiene {tam.min():,} transacciones ({menor:.3f}%).')
    print('Un cluster casi vacío significa que K-means gastó un centroide en unos pocos')
    print('valores extremos. El silhouette alto en ese caso es un artefacto, no calidad.')
    print('Revisar la sección 1.6: alguna variable sigue muy sesgada.')
else:
    print(f'Cluster más chico: {tam.min():,} transacciones ({menor:.2f}%) — sin degeneración.')

### 2.2 Perfil de los clusters

Los centroides en unidades originales dicen qué caracteriza a cada grupo. Las medias por cluster
expresadas en desviaciones respecto del promedio general permiten ver de un vistazo qué variable
distingue a cada uno.

In [ ]:
centros = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                       columns=FEATURES, index=[f'C{i}' for i in range(K_KMEANS)])
print('--- Centroides en el espacio transformado ---')
print(centros.round(2).to_string())

# En unidades reales: promedio de los valores originales por cluster
cols_r = [c for c in df_real.columns]
tmp = df_real.copy(); tmp['_g'] = df['cluster_km'].values
centros_reales = tmp.groupby('_g')[cols_r].mean()
centros_reales.index = [f'C{i}' for i in centros_reales.index]
print('\n--- Promedios en unidades reales (pesos, ítems, segundos) ---')
print(centros_reales.round(2).to_string())
del tmp; _ = gc.collect()

z_km = df.groupby('cluster_km')[FEATURES].mean()
z_km = (z_km - df[FEATURES].mean()) / df[FEATURES].std()
z_km.index = [f'C{i}' for i in z_km.index]

plt.figure(figsize=(max(9, len(FEATURES)*0.9), 0.7*K_KMEANS + 2.5))
sns.heatmap(z_km, cmap='RdBu_r', center=0, annot=True, fmt='.2f', linewidths=.5,
            cbar_kws={'label': 'desviaciones del promedio'}, annot_kws={'size': 8})
plt.title(f'Perfil de los {K_KMEANS} clusters de K-means')
plt.ylabel('Cluster'); plt.tight_layout(); plt.show()

print('=== RASGOS DISTINTIVOS ===\n')
for c in z_km.index:
    fila = z_km.loc[c].sort_values(key=abs, ascending=False).head(5)
    n = int((df['cluster_km'] == int(c[1:])).sum())
    print(f'{c}  |  {n:,} transacciones ({n/len(df)*100:.1f}%)')
    for var, v in fila.items():
        print(f'    {var:26s} {v:+.2f} desviaciones')
    print()

### 2.3 Descripción en lenguaje simple

El Problema 2 pide explicar los clusters a alguien sin conocimiento técnico. Un mapa de calor de
desviaciones no sirve para eso: un jefe de local no sabe qué es una desviación estándar.

La celda siguiente redacta un párrafo por cluster con los promedios en pesos, ítems y minutos. Es
un borrador que hay que editar, no el texto final.

In [ ]:
def _pesos(x):
    return f'${x:,.0f}'.replace(',', '.')

def _dur(seg):
    seg = float(seg)
    if seg < 60:   return f'{seg:.0f} s'
    if seg < 3600: return f'{int(seg//60)} min {int(seg%60)} s'
    return f'{seg/3600:.1f} h'

def describir_clusters(label_col, prefijo='C'):
    """Redacta una descripción por cluster con promedios en unidades reales.

    Usa df_real, que guarda los valores ANTES de Box-Cox y arcsine, para que los
    números estén en pesos, ítems y segundos y no en la escala transformada.
    """
    cols = [c for c in df_real.columns]
    tmp = df_real.copy(); tmp['_g'] = df[label_col].values
    d = tmp.groupby('_g')
    perfil, glob, n = d[cols].mean(), df_real[cols].mean(), d.size()
    lineas = []
    for g in perfil.index:
        p = perfil.loc[g]
        nombre = 'RUIDO' if g == -1 else f'{prefijo}{g}'
        t = [f'**{nombre}** — {n[g]:,} transacciones ({100*n[g]/len(df):.1f}% del total).']
        partes = []
        if 'MontoTicket' in p:  partes.append(f'ticket de {_pesos(p.MontoTicket)}')
        if 'CantItem' in p:     partes.append(f'{p.CantItem:.0f} ítems')
        if 'duracion_seg' in p: partes.append(f'atendida en {_dur(p.duracion_seg)}')
        if partes: t.append('Compra promedio: ' + ', '.join(partes) + '.')
        if 'ratio_item_anulado' in p:
            comp = 'por encima' if p.ratio_item_anulado > glob.ratio_item_anulado else 'por debajo'
            t.append(f'Se anula el {100*p.ratio_item_anulado:.0f}% de los ítems '
                     f'(promedio general {100*glob.ratio_item_anulado:.0f}%), {comp} de lo normal.')
        if 'ratio_monto_anulado' in p:
            t.append(f'En dinero se anula el {100*p.ratio_monto_anulado:.0f}% '
                     f'(promedio general {100*glob.ratio_monto_anulado:.0f}%)'
                     + (f', unos {_pesos(p.Monto_Anul)} por transacción.' if 'Monto_Anul' in p else '.'))
        if 'ratio_efectivo' in p:
            t.append(f'Paga {100*p.ratio_efectivo:.0f}% en efectivo '
                     f'(promedio general {100*glob.ratio_efectivo:.0f}%).')
        if 'hora_decimal' in p:
            h = p.hora_decimal
            t.append(f'Ocurre en promedio a las {int(h):02d}:{int((h%1)*60):02d}.')
        lineas.append(' '.join(t))
    return lineas

print('='*86)
print('DESCRIPCIÓN DE LOS CLUSTERS EN LENGUAJE SIMPLE — K-means')
print('='*86)
for l in describir_clusters('cluster_km'):
    print('\n' + l)
print('\n' + '='*86)
print('Borrador con los promedios reales. Hay que editarlo: ponerle a cada grupo un nombre')
print('de negocio y decir qué debería hacer el supermercado con él.')

### 2.4 Caracterización con variables que NO entraron al modelo

Si los grupos formados solo con variables de comportamiento resultan además distintos en las dummies
de medio de pago o en las categorías de producto, la segmentación captura algo real y no un
artefacto del preprocesamiento.

In [ ]:
cand = [c for c in dict.fromkeys(PARA_PERFILAR) if c in df.columns and c not in FEATURES]
disc = {}
for c in cand:
    m, s = df.groupby('cluster_km')[c].mean(), df[c].std()
    if s > 1e-9: disc[c] = float((m.max() - m.min()) / s)
desc_cols = [c for c, _ in sorted(disc.items(), key=lambda kv: -kv[1])[:15]]

print('Variables externas más discriminantes:')
for c in desc_cols:
    print(f'   {c:30s} rango de medias = {disc[c]:.2f} desviaciones')

perfil_ext = df.groupby('cluster_km')[desc_cols].mean()
print('\n--- Promedio por cluster ---')
print(perfil_ext.round(3).to_string())

plt.figure(figsize=(max(9, len(desc_cols)*0.8), 0.7*K_KMEANS + 2.5))
z_ext = (perfil_ext - df[desc_cols].mean()) / (df[desc_cols].std() + 1e-9)
sns.heatmap(z_ext, cmap='PuOr_r', center=0, annot=True, fmt='.2f', linewidths=.5,
            annot_kws={'size': 8})
plt.title('Validación: variables fuera del modelo')
plt.tight_layout(); plt.show()

In [ ]:
# ¿Los clusters se concentran en ciertos locales, terminales o supervisores?
for c in ['LocalFisicoCod','TerminalId','SupervisorCod']:
    if c in df.columns:
        conc = df.groupby('cluster_km')[c].nunique()
        top  = df.groupby('cluster_km')[c].agg(lambda s: s.value_counts().index[0])
        pct  = df.groupby('cluster_km')[c].agg(lambda s: s.value_counts(normalize=True).iloc[0]*100)
        print(f'\n--- {c} ---')
        print(pd.DataFrame({'valores_distintos': conc, 'mas_frecuente': top,
                            '%_del_cluster': pct.round(1)}).to_string())

### 2.5 Visualización

Único uso de PCA en el notebook, y solo para dibujar en dos dimensiones. La clase 02 lo presenta
como herramienta de reducción de dimensionalidad; aquí cumple ese rol únicamente para el gráfico.
Los modelos trabajan sobre las variables originales.

In [ ]:
pca_viz = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_m)
V = pca_viz.transform(X_m)
lab_m = kmeans.predict(X_m)

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))

ax[0].scatter(V[:, 0], V[:, 1], c=lab_m, cmap='tab10', s=3, alpha=.35)
cen = pca_viz.transform(kmeans.cluster_centers_)
ax[0].scatter(cen[:, 0], cen[:, 1], c='black', marker='X', s=260,
              edgecolors='white', linewidths=2)
for i, (x, y) in enumerate(cen):
    ax[0].annotate(f'C{i}', (x, y), fontsize=12, fontweight='bold',
                   xytext=(7, 7), textcoords='offset points')
ax[0].set_title(f'K-means (K={K_KMEANS}) — vista en 2 componentes')
ax[0].set_xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]*100:.1f}% var)')
ax[0].set_ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]*100:.1f}% var)')

sub_i = np.random.choice(len(X_m), min(15_000, len(X_m)), replace=False)
sv = silhouette_samples(X_m[sub_i], lab_m[sub_i])
y0 = 10
for i in range(K_KMEANS):
    vals = np.sort(sv[lab_m[sub_i] == i])
    if len(vals) == 0: continue
    ax[1].fill_betweenx(np.arange(y0, y0+len(vals)), 0, vals,
                        color=plt.cm.tab10(i/10), alpha=.75)
    ax[1].text(-0.06, y0 + len(vals)/2, f'C{i}', fontsize=10)
    y0 += len(vals) + 10
ax[1].axvline(sv.mean(), c='crimson', ls='--', label=f'media {sv.mean():.3f}')
ax[1].set_title('Silhouette por cluster'); ax[1].set_xlabel('coeficiente')
ax[1].set_yticks([]); ax[1].legend()

plt.tight_layout(); plt.show()

### 2.6 Interpretación de negocio

> **Redactar aquí.** Un párrafo por cluster, sin jerga, partiendo del borrador de 2.3 y de los
> rasgos de 2.2 y 2.4: a qué tipo de transacción corresponde, qué lo hace distinto, y si merece
> atención del área de prevención de pérdidas.
>
> - **C0**: …
> - **C1**: …
> - **C2**: …
> - **C3**: …

### 2.7 Evaluación supervisada con la etiqueta `Fraude`

La clase 06 define la evaluación supervisada como medir el ajuste de los clusters en datos que
tienen etiquetas, y propone tres métricas: **Rand index**, **Normalized Mutual Information** y
**homogeneidad, completitud y V-measure**. También señala que a cada cluster se le puede asignar una
etiqueta según las etiquetas que contiene.

Como `Fraude` no se usó para construir los clusters, esta es una prueba honesta: si los grupos
formados solo con variables de comportamiento concentran el fraude conocido, el modelo encontró
algo real.

Los valores `-1` corresponden a transacciones sin revisar, así que la tasa se calcula solo sobre
las revisadas: contar los `-1` como limpios subestimaría la tasa.

In [ ]:
if ETIQUETA and ETIQUETA in df.columns:
    print('Distribución de la etiqueta:')
    print(df[ETIQUETA].value_counts().sort_index().to_string())
    print('\n  -1 = sin revisar   0 = revisada sin fraude   1 = fraude confirmado')

    marcadas = df[ETIQUETA].isin([0, 1])
    base = (df.loc[marcadas, ETIQUETA] == 1).mean()*100
    print(f'\nRevisadas: {int(marcadas.sum()):,} de {len(df):,} ({marcadas.mean()*100:.1f}%)')
    print(f'Tasa de fraude sobre lo revisado: {base:.3f}%\n')

    # Etiqueta asignada a cada cluster según las etiquetas que contiene (clase 06)
    tab = df.groupby('cluster_km')[ETIQUETA].agg(
            n='size',
            pct_fraude=lambda s: (s[s.isin([0,1])] == 1).mean()*100 if s.isin([0,1]).any() else 0.0,
            pct_revisadas=lambda s: s.isin([0,1]).mean()*100)
    tab['etiqueta_asignada'] = np.where(tab.pct_fraude > base, 'FRAUDE', 'sin fraude')
    tab.index = [f'C{i}' for i in tab.index]
    print('=== Tasa de fraude por cluster ===')
    print(tab.round(3).to_string())

    plt.figure(figsize=(8, 4))
    col = ['crimson' if v > base else 'steelblue' for v in tab.pct_fraude]
    plt.bar(tab.index, tab.pct_fraude, color=col)
    plt.axhline(base, ls='--', c='black', lw=1, label=f'tasa general {base:.2f}%')
    plt.ylabel('% de fraude confirmado'); plt.title('Fraude por cluster')
    plt.legend(); plt.tight_layout(); plt.show()

    y_true, y_pred = df.loc[marcadas, ETIQUETA], df.loc[marcadas, 'cluster_km']
    h, comp, v = homogeneity_completeness_v_measure(y_true, y_pred)
    print(f'\n=== Métricas de la clase 06, sobre {int(marcadas.sum()):,} transacciones ===')
    print('Rand index               :', round(rand_score(y_true, y_pred), 4))
    print('Normalized Mutual Info   :', round(normalized_mutual_info_score(y_true, y_pred), 4))
    print('Homogeneidad             :', round(h, 4), ' (cada cluster contiene una sola clase)')
    print('Completitud              :', round(comp, 4), ' (cada clase cae en un solo cluster)')
    print('V-measure                :', round(v, 4), ' (media armónica de ambas)')
    print('\nLa homogeneidad es la relevante aquí: mide si los clusters son puros. La')
    print('completitud será baja por construcción, porque el fraude se reparte entre varios')
    print('grupos de comportamiento, y eso no es un defecto del modelo.')
else:
    print('Sin etiqueta externa disponible.')

---
# Problema 3 — DBSCAN (2 ptos)

DBSCAN agrupa por **densidad**: un punto es *core* si tiene al menos `MinPts` vecinos dentro del
radio `eps`; es *borde* si cae dentro del radio de un core sin serlo; y es **ruido** si no es
ninguna de las dos cosas.

Por qué encaja con este problema:

- No obliga a que toda transacción pertenezca a un grupo. Las que no siguen ningún patrón común
  quedan como `-1`. En un caso de fraude ese grupo es el objetivo, no un descarte.
- No impone forma esférica a los clusters, a diferencia de K-means.
- El número de clusters emerge de la densidad en vez de fijarse de antemano.

El costo es computacional, así que se ajusta sobre una muestra. **Los resultados de DBSCAN quedan
referidos a esa muestra**, y el Problema 4 compara ambos modelos sobre ella para que la comparación
sea sobre los mismos puntos.

### 3.1 Selección de `eps` y `MinPts`

El procedimiento de la clase 07:

- Para `MinPts`, la regla base es `MinPts >= dim + 1`.
- Para `eps`, se calcula la distancia de cada punto a su k-ésimo vecino con
  `sklearn.neighbors.NearestNeighbors`, se ordenan de menor a mayor y se grafica. El valor de `eps`
  se **lee del gráfico**: está donde la curva empieza a crecer de forma acelerada. Los puntos por
  sobre ese quiebre son los atípicos.

En el ejemplo de la clase, la lectura del gráfico daba un rango (`eps ≈ 2.2 a 3.0`), no un valor
único. Aquí se hace lo mismo: se lee un rango y se prueban valores dentro de él.

In [ ]:
DIM = X_scaled.shape[1]
MIN_PTS = max(2*DIM, DIM + 1)      # regla MinPts >= dim+1, con margen por el tamaño de la base
print(f'Dimensiones: {DIM}  ->  MinPts = {MIN_PTS}')

N_DBSCAN = 30_000
idx_db = np.random.choice(len(X_scaled), min(N_DBSCAN, len(X_scaled)), replace=False)
X_db = X_scaled[idx_db]
print('Muestra para DBSCAN:', X_db.shape)

nn = NearestNeighbors(n_neighbors=MIN_PTS).fit(X_db)
dist, _ = nn.kneighbors(X_db)
d_k = np.sort(dist[:, -1])         # distancia al MinPts-ésimo vecino, ordenada

plt.figure(figsize=(11, 5))
plt.plot(d_k, lw=1.6)
plt.xlabel('Puntos ordenados'); plt.ylabel(f'Distancia al vecino {MIN_PTS}')
plt.title(f'Curva de distancia al k-ésimo vecino (MinPts={MIN_PTS})')
plt.grid(alpha=.3)
for p in [50, 75, 90, 95, 99]:
    v = np.percentile(d_k, p)
    plt.axhline(v, ls=':', c='gray', lw=.8)
    plt.text(len(d_k)*1.002, v, f'p{p}={v:.2f}', fontsize=8, va='center')
plt.tight_layout(); plt.show()

print('Percentiles de la distancia al k-ésimo vecino:')
for p in [50, 75, 90, 95, 99]:
    print(f'   p{p:2d} = {np.percentile(d_k, p):.4f}')
print('\nLeer del gráfico el punto donde la curva se dispara. Ese es el rango de eps a probar.')

### 3.2 Prueba de valores de `eps`

La clase 07 muestra que DBSCAN es sensible a `eps`: con `MinPts=4`, los valores 9,75 y 9,92 dan
resultados distintos. Por eso no basta con un valor: se prueban varios dentro del rango leído del
gráfico y se compara qué produce cada uno.

Lo que se busca: un número de clusters interpretable y una proporción de ruido compatible con lo
que el negocio considera plausible como tasa de transacciones anómalas.

In [ ]:
candidatos = sorted({round(float(np.percentile(d_k, p)), 3)
                     for p in [40, 50, 60, 70, 80, 85, 90, 95]})
print('Valores de eps a probar (leídos de la curva):', candidatos, '\n')

grid = []
for e in candidatos:
    for mp in sorted({max(3, MIN_PTS//2), MIN_PTS, MIN_PTS*2}):
        lab = DBSCAN(eps=e, min_samples=mp, metric='euclidean', n_jobs=-1).fit_predict(X_db)
        n_c = len(set(lab)) - (1 if -1 in lab else 0)
        fila = {'eps': e, 'MinPts': mp, 'n_clusters': n_c,
                'ruido_%': round((lab == -1).mean()*100, 2), 'silhouette': np.nan}
        if n_c >= 2:
            ok = lab != -1
            if ok.sum() > n_c:
                fila['silhouette'] = round(silhouette_score(X_db[ok], lab[ok]), 4)
        grid.append(fila)

g = pd.DataFrame(grid)
print(g.to_string(index=False))
print('\nCómo leer la tabla:')
print('  - Si todo da 1 cluster y ~0% de ruido, eps es demasiado grande.')
print('  - Si todo da mucho ruido y clusters diminutos, eps es demasiado chico.')
print('  - Buscar un tramo donde el nº de clusters se mantenga al cambiar eps: es más robusto.')
print('  - El silhouette de DBSCAN se calcula sin el ruido, así que no es comparable')
print('    directamente con el de K-means; hay que mirarlo junto al % de ruido.')

> **Decisión sobre `eps` y `MinPts`.** Fijar abajo según la tabla y justificar por escrito.

In [ ]:
EPS = float(np.percentile(d_k, 70))   # <-- AJUSTAR según la tabla de 3.2
MIN_SAMPLES = MIN_PTS                 # <-- AJUSTAR

db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric='euclidean', n_jobs=-1).fit(X_db)
lab_db = db.labels_

n_cl = len(set(lab_db)) - (1 if -1 in lab_db else 0)
n_ruido = int((lab_db == -1).sum())
print(f'eps = {EPS:.4f}   MinPts = {MIN_SAMPLES}')
print(f'Clusters : {n_cl}')
print(f'Ruido    : {n_ruido:,} ({n_ruido/len(lab_db)*100:.2f}%)')
print(f'Core     : {len(db.core_sample_indices_):,}')

vc = pd.Series(lab_db).value_counts().sort_index()
tab_db = pd.DataFrame({'n': vc, '%': (vc/len(lab_db)*100).round(2)})
print('\nTamaño de cada grupo:')
print(tab_db.to_string())

if n_cl <= 1:
    print('\nDBSCAN encontró un solo cluster: no hay segmentación, funciona como')
    print('detector de anomalías. El entregable es el grupo de ruido.')
else:
    mayor = tab_db.loc[tab_db.index != -1, '%'].max()
    if mayor > 90:
        print(f'\nUn cluster concentra {mayor:.1f}%: DBSCAN detecta anomalías más que segmentar.')
    else:
        print(f'\nEl cluster mayor tiene {mayor:.1f}%: hay segmentación real.')

# Etiquetas de DBSCAN sobre la muestra, y K-means sobre los mismos puntos
muestra_db = df.iloc[idx_db].copy()
muestra_db['cluster_db'] = lab_db
muestra_db['cluster_km_m'] = kmeans.predict(X_db)
print(f'\nmuestra_db: {muestra_db.shape[0]:,} transacciones con ambas etiquetas.')

### 3.3 Perfil de los grupos

El grupo `-1` se analiza como cualquier otro: es el que concentra las transacciones que no siguen
ningún patrón denso, y por lo tanto el candidato natural a revisión manual.

In [ ]:
z_db = muestra_db.groupby('cluster_db')[FEATURES].mean()
z_db = (z_db - df[FEATURES].mean()) / df[FEATURES].std()
z_db.index = ['RUIDO (-1)' if i == -1 else f'D{i}' for i in z_db.index]

plt.figure(figsize=(max(9, len(FEATURES)*0.9), 0.7*len(z_db) + 2.5))
sns.heatmap(z_db, cmap='RdBu_r', center=0, annot=True, fmt='.2f', linewidths=.5,
            cbar_kws={'label': 'desviaciones del promedio'}, annot_kws={'size': 8})
plt.title('Perfil de los grupos de DBSCAN')
plt.tight_layout(); plt.show()

tmp = df_real.iloc[idx_db].copy(); tmp['_g'] = muestra_db['cluster_db'].values
perfil_db = tmp.groupby('_g')[list(df_real.columns)].mean()
perfil_db.index = ['RUIDO' if i == -1 else f'D{i}' for i in perfil_db.index]
print('--- Promedios en unidades reales (pesos, ítems, segundos) ---')
print(perfil_db.round(2).to_string())
del tmp; _ = gc.collect()

# ¿Algún grupo pequeño está definido por una variable saturada, y no por una conducta?
print('\n--- Chequeo de saturación en grupos pequeños ---')
for gr in sorted(set(lab_db)):
    n_g = int((lab_db == gr).sum())
    if gr == -1 or n_g > len(lab_db)*0.10: continue
    sub = muestra_db.loc[muestra_db.cluster_db == gr, FEATURES]
    sat = {c: round(float((sub[c] > 0.99).mean()), 3)
           for c in FEATURES if c.startswith('ratio_') and (sub[c] > 0.99).mean() > 0.5}
    cero = {c: round(float((sub[c].abs() < 1e-6).mean()), 3)
            for c in FEATURES if (sub[c].abs() < 1e-6).mean() > 0.9}
    d = {**{f'{k}=1': v for k, v in sat.items()}, **{f'{k}=0': v for k, v in cero.items()}}
    print(f'  D{gr} (n={n_g:,}): ' + (f'{d}  <- artefacto, no conducta' if d
                                      else 'sin saturación evidente'))

In [ ]:
V_db = pca_viz.transform(X_db)
plt.figure(figsize=(8, 6.5))
ruido_m = lab_db == -1
plt.scatter(V_db[~ruido_m, 0], V_db[~ruido_m, 1], c=lab_db[~ruido_m],
            cmap='tab10', s=5, alpha=.45)
plt.scatter(V_db[ruido_m, 0], V_db[ruido_m, 1], c='black', s=7, alpha=.55,
            marker='x', label=f'ruido ({int(ruido_m.sum()):,})')
plt.title(f'DBSCAN — eps={EPS:.3f}, MinPts={MIN_SAMPLES}')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend()
plt.tight_layout(); plt.show()

### 3.4 Descripción en lenguaje simple e interpretación

In [ ]:
def describir_db():
    cols = [c for c in df_real.columns]
    tmp = df_real.iloc[idx_db].copy(); tmp['_g'] = muestra_db['cluster_db'].values
    d = tmp.groupby('_g')
    perfil, glob, n = d[cols].mean(), df_real[cols].mean(), d.size()
    out = []
    for gr in perfil.index:
        p = perfil.loc[gr]
        nombre = 'RUIDO' if gr == -1 else f'D{gr}'
        t = [f'**{nombre}** — {n[gr]:,} de la muestra ({100*n[gr]/len(muestra_db):.1f}%).']
        partes = []
        if 'MontoTicket' in p:  partes.append(f'ticket de {_pesos(p.MontoTicket)}')
        if 'CantItem' in p:     partes.append(f'{p.CantItem:.0f} ítems')
        if 'duracion_seg' in p: partes.append(f'atendida en {_dur(p.duracion_seg)}')
        if partes: t.append('Compra promedio: ' + ', '.join(partes) + '.')
        if 'ratio_item_anulado' in p:
            comp = 'por encima' if p.ratio_item_anulado > glob.ratio_item_anulado else 'por debajo'
            t.append(f'Se anula el {100*p.ratio_item_anulado:.0f}% de los ítems '
                     f'(promedio general {100*glob.ratio_item_anulado:.0f}%), {comp} de lo normal.')
        if 'ratio_efectivo' in p:
            t.append(f'Paga {100*p.ratio_efectivo:.0f}% en efectivo '
                     f'(promedio general {100*glob.ratio_efectivo:.0f}%).')
        out.append(' '.join(t))
    return out

print('='*86)
print('DESCRIPCIÓN DE LOS GRUPOS EN LENGUAJE SIMPLE — DBSCAN')
print('='*86)
for l in describir_db():
    print('\n' + l)

> **Redactar aquí la interpretación de negocio de DBSCAN.** Un párrafo por grupo, incluyendo
> explícitamente el de ruido: cuántas transacciones son, qué las caracteriza y qué debería hacer el
> supermercado con ellas.

### 3.5 Evaluación supervisada del grupo de ruido

DBSCAN no vio la etiqueta. Si el grupo de ruido concentra fraude por encima de la tasa general, el
método está funcionando como detector de anomalías aunque no sirva para segmentar.

In [ ]:
if ETIQUETA and ETIQUETA in muestra_db.columns:
    marc = muestra_db[ETIQUETA].isin([0, 1])
    base_m = (muestra_db.loc[marc, ETIQUETA] == 1).mean()*100
    print(f'Tasa de fraude en la muestra, sobre lo revisado: {base_m:.3f}%\n')

    tab = muestra_db.groupby('cluster_db')[ETIQUETA].agg(
            n='size',
            pct_fraude=lambda s: (s[s.isin([0,1])] == 1).mean()*100 if s.isin([0,1]).any() else 0.0,
            pct_revisadas=lambda s: s.isin([0,1]).mean()*100)
    tab['etiqueta_asignada'] = np.where(tab.pct_fraude > base_m, 'FRAUDE', 'sin fraude')
    tab.index = ['RUIDO (-1)' if i == -1 else f'D{i}' for i in tab.index]
    print(tab.round(3).to_string())

    if marc.any() and len(set(muestra_db.cluster_db)) > 1:
        y_t, y_p = muestra_db.loc[marc, ETIQUETA], muestra_db.loc[marc, 'cluster_db']
        h, comp, v = homogeneity_completeness_v_measure(y_t, y_p)
        print('\n=== Métricas de la clase 06 ===')
        print('Rand index             :', round(rand_score(y_t, y_p), 4))
        print('Normalized Mutual Info :', round(normalized_mutual_info_score(y_t, y_p), 4))
        print('Homogeneidad           :', round(h, 4))
        print('Completitud            :', round(comp, 4))
        print('V-measure              :', round(v, 4))

---
# Problema 4 — Comparación y recomendación (1 pto)

La comparación se hace sobre la **misma muestra**, donde ambos modelos tienen etiquetas, para que
sea sobre los mismos puntos.

### 4.1 Evaluación no supervisada

Para DBSCAN el silhouette se calcula **excluyendo el ruido**, porque el grupo `-1` no es un cluster
sino un residuo. Por eso el valor no es directamente comparable con el de K-means, y hay que leerlo
junto a la columna de transacciones sin asignar.

In [ ]:
lab_km_db = muestra_db['cluster_km_m'].values
filas = [{'modelo': f'K-means (K={K_KMEANS})', 'n_grupos': K_KMEANS, 'sin_asignar_%': 0.0,
          'silhouette': round(silhouette_score(X_db, lab_km_db), 4)}]

ok = lab_db != -1
if len(set(lab_db[ok])) >= 2:
    filas.append({'modelo': f'DBSCAN (eps={EPS:.3f})', 'n_grupos': n_cl,
                  'sin_asignar_%': round((~ok).mean()*100, 2),
                  'silhouette': round(silhouette_score(X_db[ok], lab_db[ok]), 4)})
else:
    filas.append({'modelo': f'DBSCAN (eps={EPS:.3f})', 'n_grupos': n_cl,
                  'sin_asignar_%': round((~ok).mean()*100, 2), 'silhouette': np.nan})

comparacion = pd.DataFrame(filas)
print(comparacion.to_string(index=False))
print('\nSSE de K-means (inercia):', round(kmeans.inertia_, 1))
print('DBSCAN no minimiza una función de costo, así que no tiene un SSE equivalente.')

### 4.2 ¿Están viendo lo mismo los dos modelos?

Rand index y NMI, ambas de la clase 06, miden el acuerdo entre dos particiones. Cerca de 0 significa
que cada modelo captura una estructura distinta; cerca de 1, que encontraron lo mismo por caminos
distintos. Los dos resultados son informativos y hay que interpretarlos, no solo reportarlos.

In [ ]:
km_m, db_m = muestra_db['cluster_km_m'].values, muestra_db['cluster_db'].values

print('Rand index             :', round(rand_score(km_m, db_m), 4))
print('Normalized Mutual Info :', round(normalized_mutual_info_score(km_m, db_m), 4))

ct = pd.crosstab(km_m, db_m)
ct.index.name = 'K-means'; ct.columns.name = 'DBSCAN'
print('\n--- Tabla cruzada ---')
print(ct.to_string())

if ct.shape[1] > 1:
    plt.figure(figsize=(max(6, ct.shape[1]*1.1), 0.6*ct.shape[0] + 2.5))
    sns.heatmap(ct.div(ct.sum(axis=1), axis=0)*100, annot=True, fmt='.1f',
                cmap='Blues', cbar_kws={'label': '% de la fila'})
    plt.title('Cómo se reparte cada cluster de K-means dentro de DBSCAN')
    plt.tight_layout(); plt.show()

if (db_m == -1).any():
    r = pd.Series(km_m[db_m == -1]).value_counts(normalize=True).sort_index()*100
    print('\nDistribución del ruido de DBSCAN entre los clusters de K-means (%):')
    print(r.round(1).to_string())
    print('\nSi el ruido se reparte parejo, K-means lo diluye y no lo detectaría.')
    print('Si se concentra en un cluster, ese es el equivalente aproximado en K-means.')

### 4.3 Cuadro comparativo

In [ ]:
criterios = pd.DataFrame([
 ['Número de grupos',                'Se fija a priori (K)',        'Emerge de la densidad'],
 ['Forma de los clusters',           'Esférica, tamaños similares', 'Arbitraria'],
 ['Parámetros a definir',            'K',                           'eps y MinPts'],
 ['Sensibilidad a los parámetros',   'Baja',                        'Alta (clase 07)'],
 ['Transacciones sin asignar',       'Ninguna',                     'Las etiquetadas como ruido'],
 ['Detecta transacciones atípicas',  'No, todas caen en un cluster','Sí, grupo de ruido -1'],
 ['Etiqueta casos nuevos',           'Sí, centroide más cercano',   'No de forma nativa'],
 ['Costo computacional',             'Bajo',                        'Alto'],
 ['Interpretación para el negocio',  'Directa vía centroides',      'Requiere perfilar cada grupo'],
], columns=['Criterio', 'K-means', 'DBSCAN'])
print(criterios.to_string(index=False))

### 4.4 Recomendación

**Redactar con los números propios.** Las líneas argumentales que los resultados permiten sostener:

**K-means**, si el objetivo es una segmentación estable que corra sobre toda la base y alimente un
tablero de monitoreo. Escala, etiqueta casos nuevos con una regla simple y sus centroides se
explican sin jerga. El costo: no distingue la transacción atípica, la asigna al cluster más cercano
de todas formas.

**DBSCAN**, si el objetivo es priorizar casos para revisión manual. El grupo de ruido entrega
directamente la lista de transacciones que no siguen ningún patrón, que es lo que el área de
prevención de pérdidas puede accionar. El costo: no escala sin muestrear y no etiqueta casos nuevos
de forma nativa, tal como señala la clase 07.

**Los dos en conjunto**, que suele ser lo correcto operativamente: K-means define los segmentos de
comportamiento normal y corre en producción; DBSCAN se ejecuta periódicamente sobre muestras para
levantar los casos a auditar.

Justificar con los números concretos de 4.1 y 4.2, y sobre todo con la evaluación supervisada de
2.7 y 3.5: si un grupo concentra el fraude conocido muy por encima de la tasa general, ese es el
argumento más fuerte del informe, porque demuestra que el método encontró el patrón sin que nadie
se lo enseñara.

Si ningún grupo concentra fraude, la conclusión honesta es que las variables disponibles no separan
el fraude marcado, y eso también es un resultado: indicaría que el fraude de esta base no se
distingue por el patrón de anulación sino por algo que no está medido.

---
## Exportación de resultados

In [ ]:
SALIDA = '/content/drive/MyDrive/tarea1_resultados'
os.makedirs(SALIDA, exist_ok=True)
for f in os.listdir(SALIDA):
    if 'gmm' in f.lower():
        os.remove(os.path.join(SALIDA, f)); print('Eliminado archivo obsoleto:', f)

cols = [c for c in ['Llave','cluster_km', ETIQUETA] if c and c in df.columns]
df[cols].to_csv(f'{SALIDA}/etiquetas_kmeans.csv', index=False)

cols_db = [c for c in ['Llave','cluster_db','cluster_km_m', ETIQUETA]
           if c and c in muestra_db.columns]
muestra_db[cols_db].to_csv(f'{SALIDA}/etiquetas_dbscan_muestra.csv', index=False)

z_km.to_csv(f'{SALIDA}/perfil_kmeans.csv')
z_db.to_csv(f'{SALIDA}/perfil_dbscan.csv')
centros.to_csv(f'{SALIDA}/centroides_kmeans.csv')
comparacion.to_csv(f'{SALIDA}/comparacion_modelos.csv', index=False)
met.to_csv(f'{SALIDA}/seleccion_k.csv', index=False)
g.to_csv(f'{SALIDA}/grilla_dbscan.csv', index=False)
pd.Series(FEATURES).to_csv(f'{SALIDA}/variables_usadas.csv', index=False, header=['variable'])

print('\nGuardado en', SALIDA)
for f in sorted(os.listdir(SALIDA)): print('  -', f)

---
## Checklist antes de entregar

**Decisiones que hay que justificar por escrito**
- [ ] `K_KMEANS`: leer los gráficos de 2.1 y escribir el criterio (codo del SSE, peak del
      Silhouette, y cuántos grupos se pueden explicar de verdad).
- [ ] `EPS` y `MIN_SAMPLES`: leer la curva de 3.1 y la tabla de 3.2, y justificar la elección.

**Redacción**
- [ ] Interpretación de cada cluster de K-means (2.6), partiendo del borrador de 2.3.
- [ ] Interpretación de los grupos de DBSCAN (3.4), incluido el ruido.
- [ ] Recomendación del Problema 4 (4.4) apoyada en 4.1, 4.2, 2.7 y 3.5.

**Lo que distingue un buen informe**
- [ ] Reportar la evaluación supervisada con `Fraude` y dejar explícito que **no** se usó
      para modelar.
- [ ] Reportar el Hopkins de 1.9 con su matiz: mide desviación de la uniformidad, no
      separabilidad.
- [ ] Explicar la decisión sobre outliers de 1.6 y su consecuencia sobre el silhouette.
- [ ] Mencionar que los resultados de DBSCAN están referidos a la muestra, y por qué.

**Entrega**
- [ ] Presentación en ppt, key o pdf. **No se aceptan enlaces a la nube.**
- [ ] 20 de septiembre, 23:59. Tarea individual.